In [ ]:
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.neighbors import NearestNeighbors
from scipy.special import digamma
import matplotlib.pyplot as plt
import pandas as pd
from time import time
np.random.seed(42)

OUTDIR = "output_eeg_upgraded"
os.makedirs(OUTDIR, exist_ok=True)

n_sources = 64          # cortical grid sources
n_latents = 8           # effective latent dimensionality to evaluate (projected)
n_time = 2000           # time samples (reduce to 1000 for faster runs)
electrode_list = [8, 16, 32, 64, 128]  # electrode counts to sweep
leadfield_blur = 0.25   # spatial spread of leadfield
blur_sigma_sources = 0.15  # source-field smoothness
snr_db_list = [0, 10, 20]  # SNR sweep in dB
ar_rho = 0.9            # AR(1) temporal correlation for latents
noise_spatial_sigma = 0.3  # spatial noise correlation length

def make_grid_positions(n, radius=1.0, jitter=0.03):
    theta = np.linspace(0, 2*np.pi, n, endpoint=False)
    x = radius * np.cos(theta) + np.random.randn(n)*jitter
    y = radius * np.sin(theta) + np.random.randn(n)*jitter
    return np.stack([x,y], axis=1)

def rbf_cov(positions, sigma):
    d = np.linalg.norm(positions[:,None,:] - positions[None,:,:], axis=2)
    return np.exp(-0.5*(d/sigma)**2)

def make_leadfield(elec_pos, src_pos, sigma):
    d = np.linalg.norm(elec_pos[:,None,:] - src_pos[None,:,:], axis=2)
    A = np.exp(-0.5*(d/sigma)**2)
    A = A / (np.linalg.norm(A, axis=0, keepdims=True) + 1e-12)
    return A

def analytic_mi_bits(A, Sigma_x, Sigma_eps):
    n_e = A.shape[0]
    try:
        L = np.linalg.cholesky(Sigma_eps)
        Linv = np.linalg.inv(L)
        M = Linv @ (A @ Sigma_x @ A.T) @ Linv.T
        sign, logdet = np.linalg.slogdet(np.eye(n_e) + M)
        if sign <= 0:
            vals = np.linalg.eigvalsh(np.eye(n_e) + M)
            logdet = np.sum(np.log(np.clip(vals,1e-12,None)))
    except np.linalg.LinAlgError:
        M = np.linalg.pinv(Sigma_eps) @ (A @ Sigma_x @ A.T)
        sign, logdet = np.linalg.slogdet(np.eye(n_e) + M)
    return 0.5 * logdet / np.log(2)

def simulate_ar_latents(dim, n_t, rho, Q):
    X = np.zeros((n_t, dim))
    for t in range(1, n_t):
        X[t] = rho * X[t-1] + np.random.multivariate_normal(np.zeros(dim), Q)
    return X

def ksg_mi(X, Y, k=5):
    assert X.shape[0] == Y.shape[0]
    n = X.shape[0]
    data = np.hstack([X, Y])
    nn = NearestNeighbors(n_neighbors=k+1, metric='chebyshev').fit(data)
    dist, _ = nn.kneighbors(data, n_neighbors=k+1)
    eps = dist[:, k] + 1e-12
    nx = NearestNeighbors(metric='chebyshev').fit(X)
    ny = NearestNeighbors(metric='chebyshev').fit(Y)
    nx_count = nx.radius_neighbors_graph(X, radius=eps, mode='connectivity').sum(axis=1).A1 - 1
    ny_count = ny.radius_neighbors_graph(Y, radius=eps, mode='connectivity').sum(axis=1).A1 - 1
    mi = digamma(k) + digamma(n) - np.mean(digamma(nx_count+1) + digamma(ny_count+1))
    return mi

theta = np.linspace(0, 2*np.pi, n_sources, endpoint=False)
src_pos = np.stack([0.6*np.cos(theta), 0.6*np.sin(theta)], axis=1) + np.random.randn(n_sources,2)*0.005
K_src = rbf_cov(src_pos, blur_sigma_sources)
eigvals, eigvecs = np.linalg.eigh(K_src)
idx = np.argsort(eigvals)[::-1][:n_latents]
modes = eigvecs[:, idx]  # (n_sources, n_latents)

Q = np.eye(n_latents) * 0.5
X_latents = simulate_ar_latents(n_latents, n_time, ar_rho, Q)  # (n_time, n_latents)
S = X_latents @ modes.T  # (n_time, n_sources)

results = []
t0 = time()

for n_elec in electrode_list:
    elec_pos = make_grid_positions(n_elec, radius=1.0, jitter=0.03)
    A = make_leadfield(elec_pos, src_pos, leadfield_blur)  # (n_elec, n_sources)
    covS = np.cov(S, rowvar=False)
    sig_power = np.trace(A @ covS @ A.T) / n_elec
    A = A / np.sqrt(sig_power + 1e-12)
    Sigma_eps_template = rbf_cov(elec_pos, noise_spatial_sigma)
    B = A @ modes
    Sigma_x = np.cov(X_latents, rowvar=False)
    for snr_db in snr_db_list:
        snr = 10**(snr_db/10.0)
        sigma2_noise = 1.0 / snr
        Sigma_eps = Sigma_eps_template * sigma2_noise + np.eye(n_elec)*1e-8
        Y_clean = S @ A.T
        try:
            L = np.linalg.cholesky(Sigma_eps)
            noise = np.random.randn(n_time, n_elec) @ L.T
        except np.linalg.LinAlgError:
            noise = np.random.randn(n_time, n_elec) * np.sqrt(sigma2_noise)
        Y = Y_clean + noise
        mi_analytic = analytic_mi_bits(B, Sigma_x, Sigma_eps)
        pca = PCA(n_components=n_latents)
        Y_pca = pca.fit_transform(Y)
        X_tr, X_te, Y_tr, Y_te, Yp_tr, Yp_te = train_test_split(X_latents, Y, Y_pca, test_size=0.4, random_state=1)

        ridge = Ridge(alpha=1e-2, fit_intercept=False)
        ridge.fit(Y_tr, X_tr)
        Xhat_lin = ridge.predict(Y_te)
        r2_lin = float(r2_score(X_te, Xhat_lin, multioutput='variance_weighted'))

        mlp = MLPRegressor(hidden_layer_sizes=(128,), activation='relu', max_iter=400, random_state=1)
        mlp.fit(Y_tr, X_tr)
        Xhat_mlp = mlp.predict(Y_te)
        r2_mlp = float(r2_score(X_te, Xhat_mlp, multioutput='variance_weighted'))
        try:
            mi_emp_nats = ksg_mi(X_te, Yp_te, k=5)
            mi_emp = float(mi_emp_nats / np.log(2))
        except Exception:
            mi_emp = np.nan
        try:
            mi_lin = float(ksg_mi(X_te, Xhat_lin, k=5) / np.log(2))
            mi_mlp = float(ksg_mi(X_te, Xhat_mlp, k=5) / np.log(2))
        except Exception:
            mi_lin = np.nan; mi_mlp = np.nan
        results.append({
            "n_elec": n_elec, "snr_db": snr_db,
            "mi_analytic": mi_analytic, "mi_emp": mi_emp,
            "r2_lin": r2_lin, "r2_mlp": r2_mlp,
            "mi_lin": mi_lin, "mi_mlp": mi_mlp
        })

t1 = time()
df = pd.DataFrame(results)
df.to_csv(os.path.join(OUTDIR, "results_table.csv"), index=False)

# ---- Figures ----
figA, ax = plt.subplots(1,2, figsize=(8,4))
ax[0].scatter(src_pos[:,0], src_pos[:,1], s=15, c='C1')
ax[0].set_title(f"Cortical sources (n={n_sources})"); ax[0].axis('equal'); ax[0].axis('off')
elec_pos_rep = make_grid_positions(electrode_list[-1], radius=1.0, jitter=0.03)
ax[1].scatter(elec_pos_rep[:,0], elec_pos_rep[:,1], s=20, c='C0')
ax[1].set_title(f"Electrodes (n={electrode_list[-1]})"); ax[1].axis('equal'); ax[1].axis('off')
figA.tight_layout(); figA.savefig(os.path.join(OUTDIR, "figA_schematic.png"), dpi=150); plt.close(figA)

fig1, ax = plt.subplots(1,1, figsize=(6,4))
markers = {0:'o', 10:'s', 20:'^'}
for snr_db in snr_db_list:
    sub = df[df.snr_db==snr_db].groupby('n_elec').mean().reset_index()
    ax.plot(sub.n_elec, sub.mi_analytic, marker=markers[snr_db], linestyle='-', label=f"Analytic SNR={snr_db}dB")
    ax.plot(sub.n_elec, sub.mi_emp, marker=markers[snr_db], linestyle='--', label=f"Empirical SNR={snr_db}dB")
ax.set_xscale('log', base=2); ax.set_xlabel("Electrode count (log2 scale)")
ax.set_ylabel("Mutual information (bits per sample)")
ax.set_title("Analytic vs empirical MI vs electrode count"); ax.legend(fontsize=8); ax.grid(True, ls='--', lw=0.4)
fig1.tight_layout(); fig1.savefig(os.path.join(OUTDIR, "fig1_mi_vs_electrodes.png"), dpi=150); plt.close(fig1)

fig2, ax = plt.subplots(1,1, figsize=(6,4))
for n_elec in [16, 64, 128]:
    sub = df[df.n_elec==n_elec].sort_values('snr_db')
    ax.plot(sub.snr_db, sub.mi_analytic, '-o', label=f"Analytic n={n_elec}")
    ax.plot(sub.snr_db, sub.mi_emp, '--s', label=f"Empirical n={n_elec}")
ax.set_xlabel("SNR (dB)"); ax.set_ylabel("Mutual information (bits per sample)")
ax.set_title("MI vs SNR for selected electrode densities"); ax.legend(fontsize=8); ax.grid(True, ls='--', lw=0.4)
fig2.tight_layout(); fig2.savefig(os.path.join(OUTDIR, "fig2_mi_vs_snr.png"), dpi=150); plt.close(fig2)

fig3, ax = plt.subplots(1,1, figsize=(6,4))
sc = ax.scatter(df.mi_analytic, df.r2_lin, c=df.snr_db, marker='o', label='Linear R2')
ax.scatter(df.mi_analytic, df.r2_mlp, c=df.snr_db, marker='x', label='MLP R2')
ax.set_xlabel("Analytic MI (bits/sample)"); ax.set_ylabel("Decoder R^2"); ax.set_title("Decoder R^2 vs analytic MI (color = SNR dB)")
ax.legend(fontsize=8); cbar = plt.colorbar(sc); cbar.set_label("SNR (dB)")
fig3.tight_layout(); fig3.savefig(os.path.join(OUTDIR, "fig3_decoder_vs_mi.png"), dpi=150); plt.close(fig3)

fig4, ax = plt.subplots(1,1, figsize=(6,4))
im = ax.scatter(df.mi_analytic, df.mi_lin, c=df.n_elec, cmap='viridis', marker='o', label='Linear pred MI')
ax.scatter(df.mi_analytic, df.mi_mlp, c=df.n_elec, cmap='plasma', marker='x', label='MLP pred MI')
ax.set_xlabel("Analytic MI (bits/sample)"); ax.set_ylabel("Predicted-latent MI (bits/sample)")
ax.set_title("MI recovered by decoders vs analytic MI"); ax.legend(fontsize=8)
cbar = plt.colorbar(im); cbar.set_label("n_elec (color)")
fig4.tight_layout(); fig4.savefig(os.path.join(OUTDIR, "fig4_decoder_mi.png"), dpi=150); plt.close(fig4)

with open(os.path.join(OUTDIR, "run_summary.txt"), "w") as f:
    f.write(f"Elapsed time: {t1 - t0:.2f} s\n")
    f.write(df.describe().to_string())
print("Done. Outputs in folder:", OUTDIR)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

Done. Outputs in folder: output_eeg_upgraded


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

print("\n--- Running Information Bottleneck sweeps ---")

def run_vib(Y, X, beta_list, z_dim=16, n_epochs=80, lr=1e-3, device='cpu'):
    """Train VIB model for one dataset (Y -> X) across multiple β values."""
    Y_t = torch.tensor(Y, dtype=torch.float32, device=device)
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    results = []

    for beta in beta_list:
        encoder = nn.Sequential(
            nn.Linear(Y.shape[1], 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU()
        )
        mu = nn.Linear(64, z_dim)
        logvar = nn.Linear(64, z_dim)
        decoder = nn.Sequential(
            nn.Linear(z_dim, 64), nn.ReLU(),
            nn.Linear(64, X.shape[1])
        )
        encoder, mu, logvar, decoder = encoder.to(device), mu.to(device), logvar.to(device), decoder.to(device)
        opt = optim.Adam(list(encoder.parameters()) + list(mu.parameters()) +
                         list(logvar.parameters()) + list(decoder.parameters()), lr=lr)

        for epoch in range(n_epochs):
            opt.zero_grad()
            h = encoder(Y_t)
            mu_z = mu(h); logvar_z = logvar(h)
            std = torch.exp(0.5 * logvar_z)
            eps = torch.randn_like(std)
            z = mu_z + eps * std
            X_hat = decoder(z)
            recon = 0.5 * torch.sum((X_hat - X_t)**2, dim=1).mean()  # NLL surrogate
            kl = 0.5 * torch.sum(mu_z**2 + torch.exp(logvar_z) - logvar_z - 1, dim=1).mean()
            loss = recon + beta * kl
            loss.backward(); opt.step()

        with torch.no_grad():
            h = encoder(Y_t)
            mu_z = mu(h); logvar_z = logvar(h)
            kl_nats = 0.5 * torch.sum(mu_z**2 + torch.exp(logvar_z) - logvar_z - 1, dim=1)
            I_TY_bits = kl_nats.mean().item() / np.log(2)
            X_hat = decoder(mu_z)
            mse = ((X_hat - X_t)**2).mean().item()
            I_XT_bits_proxy = -0.5 * mse / np.log(2)

        results.append({
            "beta": beta,
            "I_TY_bits": I_TY_bits,
            "I_XT_bits_proxy": I_XT_bits_proxy,
            "mse": mse
        })
        print(f"    β={beta:.3g} → I(T;Y)≈{I_TY_bits:.3f} bits, I(X;T)≈{I_XT_bits_proxy:.3f} bits")

    return results

beta_list = [1e-3, 1e-2, 1e-1, 1, 10]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ib_summary = []

for n_elec in electrode_list:
    for snr_db in snr_db_list:
        sub = df[(df.n_elec==n_elec) & (df.snr_db==snr_db)]
        print(f"\n[IB] Running n_elec={n_elec}, SNR={snr_db} dB")
        elec_pos = make_grid_positions(n_elec, radius=1.0, jitter=0.03)
        A = make_leadfield(elec_pos, src_pos, leadfield_blur)
        covS = np.cov(S, rowvar=False)
        sig_power = np.trace(A @ covS @ A.T) / n_elec
        A = A / np.sqrt(sig_power + 1e-12)
        Sigma_eps_template = rbf_cov(elec_pos, noise_spatial_sigma)
        snr = 10**(snr_db/10.0)
        sigma2_noise = 1.0 / snr
        Sigma_eps = Sigma_eps_template * sigma2_noise + np.eye(n_elec)*1e-8
        Y_clean = S @ A.T
        L = np.linalg.cholesky(Sigma_eps + np.eye(n_elec)*1e-10)
        noise = np.random.randn(n_time, n_elec) @ L.T
        Y = Y_clean + noise

        ib_res = run_vib(Y, X_latents, beta_list, z_dim=16, n_epochs=60, lr=1e-3, device=device)
        for r in ib_res:
            r.update({"n_elec": n_elec, "snr_db": snr_db})
            ib_summary.append(r)
        pd.DataFrame(ib_res).to_csv(os.path.join(OUTDIR, f"ib_results_{n_elec}_{snr_db}.csv"), index=False)

ib_df = pd.DataFrame(ib_summary)
fig, axes = plt.subplots(len(snr_db_list), len(electrode_list), figsize=(14, 8), sharex=True, sharey=True)
for i, snr_db in enumerate(snr_db_list):
    for j, n_elec in enumerate(electrode_list):
        ax = axes[i, j]
        sub = ib_df[(ib_df.n_elec==n_elec) & (ib_df.snr_db==snr_db)].sort_values("I_TY_bits")
        if not sub.empty:
            ax.plot(sub.I_TY_bits, sub.I_XT_bits_proxy, '-o', lw=1.2)
        ax.set_title(f"n={n_elec}, SNR={snr_db}dB", fontsize=8)
        ax.grid(True, ls='--', lw=0.4)
        if i == len(snr_db_list)-1: ax.set_xlabel("I(T;Y) bits")
        if j == 0: ax.set_ylabel("I(X;T) proxy bits")
fig.suptitle("Empirical Information Bottleneck tradeoffs", fontsize=12)
fig.tight_layout(rect=[0,0,1,0.97])
fig.savefig(os.path.join(OUTDIR, "fig_IB_grid.png"), dpi=150)
plt.close(fig)

pd.DataFrame(ib_summary).to_csv(os.path.join(OUTDIR, "ib_summary_all.csv"), index=False)
print("\n--- IB analysis complete. Figures and CSVs saved in output_eeg_upgraded/ ---\n")



--- Running Information Bottleneck sweeps ---

[IB] Running n_elec=8, SNR=0 dB
    β=0.001 → I(T;Y)≈82.158 bits, I(X;T)≈-1.135 bits
    β=0.01 → I(T;Y)≈50.215 bits, I(X;T)≈-1.069 bits
    β=0.1 → I(T;Y)≈14.366 bits, I(X;T)≈-1.144 bits
    β=1 → I(T;Y)≈0.140 bits, I(X;T)≈-1.830 bits
    β=10 → I(T;Y)≈0.008 bits, I(X;T)≈-1.858 bits

[IB] Running n_elec=8, SNR=10 dB
    β=0.001 → I(T;Y)≈91.477 bits, I(X;T)≈-0.705 bits
    β=0.01 → I(T;Y)≈57.226 bits, I(X;T)≈-0.709 bits
    β=0.1 → I(T;Y)≈19.700 bits, I(X;T)≈-0.738 bits
    β=1 → I(T;Y)≈1.157 bits, I(X;T)≈-1.501 bits
    β=10 → I(T;Y)≈0.005 bits, I(X;T)≈-1.860 bits

[IB] Running n_elec=8, SNR=20 dB
    β=0.001 → I(T;Y)≈67.602 bits, I(X;T)≈-0.811 bits
    β=0.01 → I(T;Y)≈68.975 bits, I(X;T)≈-0.531 bits
    β=0.1 → I(T;Y)≈18.331 bits, I(X;T)≈-0.718 bits
    β=1 → I(T;Y)≈0.909 bits, I(X;T)≈-1.588 bits
    β=10 → I(T;Y)≈0.005 bits, I(X;T)≈-1.857 bits

[IB] Running n_elec=16, SNR=0 dB
    β=0.001 → I(T;Y)≈91.043 bits, I(X;T)≈-0.995 bits
    β=

In [ ]:
#simulation written by Google Gemini

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
import os
from pathlib import Path

n_s = 64       # number of cortical sources
n_l = 8        # number of latent AR(1) processes
n_e = 32       # number of scalp electrodes
rho = 0.9      # AR(1) coefficient for latents
n_steps = 400  # time steps to simulate and animate
dt = 0.01      # time step (arbitrary units)

SNR_dB = 10.0

rng = np.random.default_rng(1)

angles = np.linspace(0, 2*np.pi, n_s, endpoint=False)
src_radius = 1.0
src_xy = np.stack([src_radius*np.cos(angles), src_radius*np.sin(angles)], axis=1)  # (n_s,2)

elec_angles = np.linspace(0, 2*np.pi, n_e, endpoint=False)
elec_radius = 1.2
elec_xy = np.stack([elec_radius*np.cos(elec_angles), elec_radius*np.sin(elec_angles)], axis=1)
elec_xy += rng.normal(scale=0.02, size=elec_xy.shape)

B = rng.normal(scale=1.0, size=(n_s, n_l))  # (n_s x n_l)
kernel_sigma = 0.5
dists = np.linalg.norm(elec_xy[:, None, :] - src_xy[None, :, :], axis=2)  # (n_e, n_s)
A = np.exp(-0.5 * (dists / kernel_sigma)**2)

A = A / np.linalg.norm(A, axis=1, keepdims=True)

# --- Latent AR(1) processes initialization ---
latents = np.zeros((n_l, n_steps))
# initial latent state drawn from unit Gaussian
latents[:, 0] = rng.normal(size=n_l)

# process noise scale (latent)
latent_noise_std = np.sqrt(1 - rho**2)  # stationary variance = 1 for AR(1)
for t in range(1, n_steps):
    latents[:, t] = rho * latents[:, t-1] + rng.normal(scale=latent_noise_std, size=n_l)

# --- Source activities X over time: (n_s, n_steps) = B @ latents ---
X = B @ latents  # shape (n_s, n_steps)

# --- Determine noise level from desired SNR ---
# Compute signal power (average over sensors & time after projection): signal at electrodes before noise
Y_signal_clean = A @ X  # (n_e, n_steps)
signal_power = np.mean(Y_signal_clean**2)
SNR_linear = 10**(SNR_dB / 10.0)
noise_power = signal_power / SNR_linear
noise_std = np.sqrt(noise_power)

# --- Simulate noisy sensor readings Y ---
Y = Y_signal_clean + rng.normal(scale=noise_std, size=Y_signal_clean.shape)

# --- Prepare plotting and animation ---
out_dir = Path('/mnt/data')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'eeg_sim_animation.gif'

fig = plt.figure(figsize=(7, 5))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2], width_ratios=[1, 1])
ax_topo = fig.add_subplot(gs[0, :])          # topographic sensor display (across top)
ax_sources = fig.add_subplot(gs[1, 0])       # source traces
ax_latents = fig.add_subplot(gs[1, 1])      # latent traces

# Topomap: scatter of electrode positions with color showing amplitude
sc = ax_topo.scatter(elec_xy[:, 0], elec_xy[:, 1], s=100)
ax_topo.set_title('Simulated EEG sensors (topographic view)')
ax_topo.set_xlim(-1.5, 1.5)
ax_topo.set_ylim(-1.5, 1.5)
ax_topo.set_xticks([]); ax_topo.set_yticks([])
# draw circle for head outline
circle = plt.Circle((0,0), 1.35, fill=False)
ax_topo.add_artist(circle)

# Source traces: show a few example sources (first 6)
n_show_src = min(6, n_s)
src_lines = []
t_plot = np.arange(0)  # will be expanded in animation update
for i in range(n_show_src):
    (ln,) = ax_sources.plot([], [], label=f'Src {i+1}')
    src_lines.append(ln)
ax_sources.set_xlim(0, n_steps*dt)
ax_sources.set_ylim(np.min(X)*1.1, np.max(X)*1.1)
ax_sources.set_title('Example source signals (subset)')
ax_sources.set_xlabel('Time (arb.)')
ax_sources.legend(loc='upper right', fontsize='small')

# Latent traces: show all latents (or up to 8)
latent_lines = []
for i in range(n_l):
    (ln,) = ax_latents.plot([], [], label=f'L{i+1}')
    latent_lines.append(ln)
ax_latents.set_xlim(0, n_steps*dt)
ax_latents.set_ylim(np.min(latents)*1.1, np.max(latents)*1.1)
ax_latents.set_title('Latent AR(1) processes')
ax_latents.set_xlabel('Time (arb.)')
ax_latents.legend(loc='upper right', fontsize='small')

# Text to show SNR and step
txt = ax_topo.text(-1.3, 1.25, f'SNR = {SNR_dB:.1f} dB\nn_e={n_e}, n_s={n_s}, n_l={n_l}', fontsize=9)

# Animation update function
window = 200  # number of past samples to show in traces (for scrolling)
def update(frame):
    # frame in range(n_steps)
    # update topomap colors
    vals = Y[:, frame]
    sc.set_offsets(elec_xy)
    sc.set_array(vals)  # matplotlib will map to colormap automatically
    ax_topo.set_title(f'Simulated EEG sensors (topographic view) — t={frame}')
    txt.set_text(f'SNR = {SNR_dB:.1f} dB\nstep = {frame} / {n_steps-1}')

    # update source traces (show last `window` samples)
    tmin = max(0, frame - window + 1)
    tvec = np.arange(tmin, frame+1) * dt
    for i, ln in enumerate(src_lines):
        data = X[i, tmin:frame+1]
        ln.set_data(tvec, data)
    ax_sources.set_xlim(tvec[0] if len(tvec)>0 else 0, dt*(tmin+window))

    # update latent traces
    for i, ln in enumerate(latent_lines):
        data = latents[i, tmin:frame+1]
        ln.set_data(tvec, data)
    ax_latents.set_xlim(tvec[0] if len(tvec)>0 else 0, dt*(tmin+window))

    return [sc] + src_lines + latent_lines + [txt]

# Create the animation
anim = animation.FuncAnimation(fig, update, frames=n_steps, interval=30, blit=False)

# Save animation to GIF
anim.save(str(out_path), writer='pillow', fps=20)

print("Animation saved to:", out_path.resolve())

# Display a still frame (final) in the notebook output as preview
plt.close(fig)

# Provide path for download
out_path.resolve().as_posix()


Animation saved to: /mnt/data/eeg_sim_animation.gif


'/mnt/data/eeg_sim_animation.gif'